In [1]:
# Import Slicer modules
import slicer
from glob import glob
import os
import zipfile
import DICOMLib.DICOMUtils as utils
import DICOMScalarVolumePlugin
import vtk
import shutil
#slicer.util.addModulePath('/Applications/Slicer 3.app/Contents/Extensions-32448/SwissSkullStripper/lib/Slicer-5.6/cli-modules')

In [13]:
# Path to data
folder_path = '/Users/caryngeady/Documents/GitHub/Brain-Project/SOLID-DICOM'
folder_path = '/Users/caryngeady/Documents/GitHub/Brain-Project/Glioma meDIP Dicoms'

# Get list of subfolders and corresponding patient IDs
patientIDs = [f for f in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, f))]
imageFolders = [os.path.join(folder_path, patientID, f) for patientID in patientIDs for f in os.listdir(os.path.join(folder_path, patientID)) if os.path.isdir(os.path.join(folder_path, patientID, f)) or f.endswith('.zip')]

In [14]:
len(imageFolders)

200

In [8]:
# Function to load a file into Slicer
def loadVolumeFromPath(dicom_directory):
    """
    Load a DICOM volume from a directory of DICOM images.

    Parameters:
    dicom_directory (str): Path to the directory containing DICOM images.

    Returns:
    vtkMRMLScalarVolumeNode: The loaded volume node, or None if loading failed.
    """
    result = utils.importDicom(dicom_directory)
    scalarVolumeReader = DICOMScalarVolumePlugin.DICOMScalarVolumePluginClass()
    db = slicer.dicomDatabase

    loadedVolume = None  # Initialize the variable to store the loaded volume

    for patient in db.patients():
      for study in db.studiesForPatient(patient):
        for series in db.seriesForStudy(study):
          print('looking at series ' + series + ' for patient ' + patient)
          files = db.filesForSeries(series)
          loadable = scalarVolumeReader.examineForImport([files])[0]
          loadedVolume = scalarVolumeReader.load(loadable)  # Store the loaded volume
          if loadedVolume:
            print(f"Loaded volume: {loadedVolume.GetName()}")
          else:
            print(f"Failed to load volume for series {series}")

    # clean-up
    #del(result,scalarVolumeReader,db,patient,study,files,loadable)
    
    return loadedVolume

# Function to run n4itkbiasfieldcorrection's functionality
def runBiasFieldCorrection(volumeNode):
    # Assuming the extension has a Python-accessible API
    parameters = {}
    parameters['inputImageName'] = volumeNode.GetID()
    
    # Create a new volume node for the patientOutputVolume output
    outputVolume = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLScalarVolumeNode")
    parameters['outputImageName'] = outputVolume.GetID()  # Corrected to use outputVolume's ID
    
    # Run the n4itkbiasfieldcorrection module with the specified parameters
    slicer.cli.run(slicer.modules.n4itkbiasfieldcorrection, None, parameters, wait_for_completion=True)
    
    # Return the output label map for further processing or saving
    return outputVolume

# Function to run swissskullstripper's functionality
def runSkullStripping(volumeNode):
    # Assuming the extension has a Python-accessible API
    parameters = {}
    parameters['patientVolume'] = volumeNode.GetID()
    
    # Create a new label map volume node for the patientMaskLabel output
    outputLabelMap = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLLabelMapVolumeNode")
    parameters['patientMaskLabel'] = outputLabelMap.GetID()

    # Create a new volume node for the patientOutputVolume output
    outputVolume = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLScalarVolumeNode")
    parameters['patientOutputVolume'] = outputVolume.GetID()  # Corrected to use outputVolume's ID
    
    # Run the SwissSkullStripper module with the specified parameters
    slicer.cli.run(slicer.modules.swissskullstripper, None, parameters, wait_for_completion=True)
    
    # Return the output label map for further processing or saving
    return outputLabelMap,outputVolume

# Processing pipeline
def preprocessingPipeline(volumePath,filename):
    # load the volume from the path
    volumeNode = loadVolumeFromPath(volumePath)
    # if the volume was successfully loaded
    if volumeNode:
        # perform the bias field correction
        outputVolume = runBiasFieldCorrection(volumeNode)
        # Construct the output file path for the patientMaskLabel
        output_folder = '/Users/caryngeady/Documents/GitHub/Brain-Project/SOLID-DICOM/out'
        output_folder = '/Users/caryngeady/Documents/GitHub/Brain-Project/Glioma meDIP Dicoms/out'
        imageFilePath = os.path.join(output_folder,'images',f"{filename}_image.nii.gz")
        fileType = 'NIfTI'  # Specify the file type explicitly if necessary
        slicer.util.saveNode(outputVolume, imageFilePath, {'fileType': fileType})
        
        # perform skull stripping
        outputLabelMap,brainVolume = runSkullStripping(volumeNode)
        maskFilePath = os.path.join(output_folder,'masks',f"{filename}_brain.nii.gz")
        slicer.util.saveNode(outputLabelMap, maskFilePath, {'fileType': fileType})
        
        # Cleanup if necessary
        slicer.mrmlScene.RemoveNode(volumeNode)
        slicer.mrmlScene.RemoveNode(outputVolume)
        slicer.mrmlScene.RemoveNode(brainVolume)
        slicer.mrmlScene.RemoveNode(outputLabelMap)

In [11]:
# Main processing loop
for folder in imageFolders:

    # Isolate the patient ID from the folder path
    patientID = os.path.basename(os.path.dirname(folder))

    if folder.endswith('.zip'):

        filename = patientID + ' - ' + os.path.splitext(os.path.basename(folder))[0]
        print('Unzipping folder.')
        
        with zipfile.ZipFile(folder, 'r') as zip_ref:
            # Create a temporary folder to extract the contents of the zip file
            temp_folder = os.path.join(folder_path, 'temp')
            os.makedirs(temp_folder, exist_ok=True)
            
            # Extract all the files from the zip file to the temporary folder
            zip_ref.extractall(temp_folder)
            
            # Perform operations on the extracted files in the temporary folder
            preprocessingPipeline(temp_folder,filename)
            
            # Remove the temporary folder after the operations are done
            shutil.rmtree(temp_folder)

    else:
        if os.path.isdir(folder):
            filename = patientID + ' - ' + os.path.basename(folder) 
            preprocessingPipeline(folder,filename)

Unzipping folder.
looking at series 1.3.6.1.4.1.12201.9423235473154.141.20240415125312594.1 for patient 1
Loaded volume: 4: Unnamed Series_3
Unzipping folder.
looking at series 1.3.6.1.4.1.12201.9423235473154.141.20240415125312594.1 for patient 1
Loaded volume: 4: Unnamed Series_4
Unzipping folder.
looking at series 1.3.6.1.4.1.12201.9423235473154.141.20240415125312594.1 for patient 1
Loaded volume: 4: Unnamed Series_5
Unzipping folder.
looking at series 1.3.6.1.4.1.12201.9423235473154.141.20240415125312594.1 for patient 1
Loaded volume: 4: Unnamed Series_6
Unzipping folder.
looking at series 1.3.6.1.4.1.12201.9423235473154.141.20240415125312594.1 for patient 1
Loaded volume: 4: Unnamed Series_7
Unzipping folder.
looking at series 1.3.6.1.4.1.12201.9423235473154.141.20240415125312594.1 for patient 1
Loaded volume: 4: Unnamed Series_8
Unzipping folder.
looking at series 1.3.6.1.4.1.12201.9423235473154.141.20240415125312594.1 for patient 1
Loaded volume: 4: Unnamed Series_9
Unzipping fol

In [9]:
folder = imageFolders[0]

patientID = os.path.basename(os.path.dirname(folder))
filename = patientID + ' - ' + os.path.splitext(os.path.basename(folder))[0]
print('Unzipping folder.')

with zipfile.ZipFile(folder, 'r') as zip_ref:
    # Create a temporary folder to extract the contents of the zip file
    temp_folder = os.path.join(folder_path, 'temp')
    os.makedirs(temp_folder, exist_ok=True)
    
    # Extract all the files from the zip file to the temporary folder
    zip_ref.extractall(temp_folder)
    
    # Perform operations on the extracted files in the temporary folder
    preprocessingPipeline(temp_folder,filename)
    
    # Remove the temporary folder after the operations are done
    shutil.rmtree(temp_folder)

Unzipping folder.
looking at series 1.3.6.1.4.1.12201.9423235473154.141.20240415125312594.1 for patient 1
Loaded volume: 4: Unnamed Series_2


In [10]:
imageFolders[-1]

'/Users/caryngeady/Documents/GitHub/Brain-Project/Glioma meDIP Dicoms/3577/20 Jan 2013 - T2.zip'